In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="cDEXVij65ewKl9sj5vt7")
project = rf.workspace("mohammad-hesham").project("action-stanford40-jiggj")
version = project.version(1)
dataset = version.download("yolov11")
                
                    

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 6.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 85.3 MB/s eta 0:00:00ta 0:00:01
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Action(Stanford40)-1 in yolov11:: 100%|██████████| 5740/5740 [00:00<00:00, 10778.82it/s]


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="cDEXVij65ewKl9sj5vt7")
project = rf.workspace("mohammad-hesham").project("sitting-standing-lyingv2-apazs")
version = project.version(1)
dataset = version.download("yolov11")
                
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to sitting-standing-lyingv2-1 in yolov11:: 100%|██████████| 3946/3946 [00:01<00:00, 3554.83it/s]


In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="cDEXVij65ewKl9sj5vt7")
project = rf.workspace("mohammad-hesham").project("human-action-recognition-x6cml-32fdy")
version = project.version(1)
dataset = version.download("yolov11")
                
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Human-Action-Recognition-1 in yolov11:: 100%|██████████| 20065/20065 [00:02<00:00, 8469.55it/s] 


In [4]:
import os
import shutil
import random
from collections import Counter
from pathlib import Path
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split

In [5]:
def find_dataset_dir(possible_names):
    for name in possible_names:
        p = Path(f"/kaggle/working/{name}")
        if p.exists():
            return p
    for name in possible_names:
        clean = name.split("-")[0]
        matches = list(Path("/kaggle/working").glob(f"*{clean}*"))
        if matches:
            return matches[0]
    return Path(f"/kaggle/working/{possible_names[0]}")

DATASETS = {
    "Action(Stanford40)": {
        "base_dir": find_dataset_dir(["Action(Stanford40)-1", "Action(Stanford40)-2"]),
        "names": [
            "applauding", "drinking", "jumping", "looking_through_a_telescope",
            "phoning", "reading", "running", "smoking", "taking_photos",
            "texting_message", "waving_hands", "writing_on_a_book"
        ],
    },
    "Human-Action-Recognition": {
        "base_dir": find_dataset_dir(["Human-Action-Recognition-1", "Human-Action-Recognition-2"]),
        "names": [
            "Drinking", "Fall-Detected", "Fall_down", "Lying_down", "Nearly_fall",
            "Sit Down", "Sitting", "Standing", "Walking", "Walking_on_Stairs",
            "crawling", "falling", "sitting", "standing", "walking"
        ],
    },
    "sitting-standing-lyingv2": {
        "base_dir": find_dataset_dir(["sitting-standing-lyingv2-1", "sitting-standing-lyingv2-6"]),
        "names": ["lying", "sitting", "standing"],
    },
}

# Auto-update class names from data.yaml if available
for ds_key, cfg in DATASETS.items():
    yaml_p = cfg["base_dir"] / "data.yaml"
    if yaml_p.exists():
        try:
            with open(yaml_p, "r") as yf:
                yd = yaml.safe_load(yf)
                if "names" in yd:
                    if isinstance(yd["names"], list):
                        cfg["names"] = yd["names"]
                    elif isinstance(yd["names"], dict):
                        cfg["names"] = [yd["names"][k] for k in sorted(yd["names"].keys())]
        except Exception:
            pass


In [6]:
def count_labels_in_split(labels_dir, class_names):
    counts = Counter()
    if not labels_dir.exists():
        return counts

    for label_file in labels_dir.glob("*.txt"):
        with open(label_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_id = int(parts[0])
                    if class_id < len(class_names):
                        counts[class_names[class_id]] += 1
                    else:
                        counts[f"Unknown_ID_{class_id}"] += 1
    return counts

In [7]:
for ds_name, config in DATASETS.items():
    print(f"\n{'='*50}")
    print(f" DATASET: {ds_name}")
    print(f"{'='*50}")

    base_path = config["base_dir"]
    class_names = config["names"]
    splits = ["train", "valid", "test"]

    split_counts = {}
    total_counts = Counter()

    for split in splits:
        labels_path = base_path / split / "labels"
        counts = count_labels_in_split(labels_path, class_names)
        split_counts[split] = counts
        total_counts.update(counts)

    df = pd.DataFrame(split_counts).fillna(0).astype(int)

    for split in splits:
        if split not in df.columns:
            df[split] = 0

    df["Total"] = df.sum(axis=1)
    df = df.sort_values(by="Total", ascending=False)

    print(df.to_string())
    print(
        f"\nTotal instances across all splits for {ds_name}: {df['Total'].sum()}"
    )


 DATASET: Action(Stanford40)
                             train  valid  test  Total
jumping                        236     59     0    295
applauding                     231     57     0    288
phoning                        207     52     0    259
drinking                       205     51     0    256
running                        201     50     0    251
writing_on_a_book              197     49     0    246
reading                        196     49     0    245
smoking                        193     48     0    241
waving_hands                   168     42     0    210
looking_through_a_telescope    162     41     0    203
texting_message                154     39     0    193
taking_photos                  149     36     0    185

Total instances across all splits for Action(Stanford40): 2872

 DATASET: Human-Action-Recognition
                   train  valid  test  Total
Fall-Detected       3149    899   450   4498
sitting             1686    451   165   2302
walking             

In [8]:
OUTPUT_DIR = Path("/kaggle/working/unified_dataset")

In [9]:
DATASETS = {
    "Action(Stanford40)-2": {
        "base_dir": Path("/kaggle/working/Action(Stanford40)-2"),
        "names": [
            "applauding", "drinking", "jumping", "looking_through_a_telescope",
            "phoning", "reading", "running", "smoking", "taking_photos",
            "texting_message", "waving_hands", "writing_on_a_book"
        ],
    },
    "Human-Action-Recognition-2": {
        "base_dir": Path("/kaggle/working/Human-Action-Recognition-2"),
        "names": [
            "Drinking", "Fall-Detected", "Fall_down", "Lying_down", "Nearly_fall",
            "Sit Down", "Sitting", "Standing", "Walking", "Walking_on_Stairs",
            "crawling", "falling", "sitting", "standing", "walking"
        ],
    },
    "Sitting-Standing-Lyingv2-6": {
        "base_dir": Path("/kaggle/working/sitting-standing-lyingv2-6"),
        "names": ["lying", "sitting", "standing"],
    },
}

In [10]:
MAPPING_RULES = {
    "Fall-Detected": 0, "falling": 0, "Fall_down": 0, "Nearly_fall": 0,
    "sitting": 1, "Sitting": 1, "Sit Down": 1,
    "standing": 2, "Standing": 2,
    "walking": 3, "Walking": 3, "Walking_on_Stairs": 3, "running": 3, "jumping": 3,
    "lying": 4, "Lying_down": 4, "crawling": 4,
    "phoning": 5, "texting_message": 5, "taking_photos": 5,
    "reading": 6, "writing_on_a_book": 6, "drinking": 6, "Drinking": 6, "smoking": 6,
    "applauding": 7, "waving_hands": 7, "looking_through_a_telescope": 7
}

In [11]:
CLASS_NAMES = {
    0: "fall",
    1: "sitting",
    2: "standing",
    3: "walking_running",
    4: "lying",
    5: "phone_interaction",
    6: "desk_activity",
    7: "gestures"
}

In [12]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

In [13]:
splits = ["train", "valid", "test"]
for split in splits:
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

stats = {split: Counter() for split in splits}
processed_files = 0

In [14]:
print("Starting dataset processing and class consolidation...")

for ds_prefix, config in DATASETS.items():
    base_dir = config["base_dir"]
    orig_class_names = config["names"]
    
    idx_to_new_id = {}
    for orig_idx, orig_name in enumerate(orig_class_names):
        if orig_name in MAPPING_RULES:
            idx_to_new_id[orig_idx] = MAPPING_RULES[orig_name]
        else:
            print(f"Warning: '{orig_name}' in {ds_prefix} has no mapping defined!")

    for split in splits:
        labels_dir = base_dir / split / "labels"
        images_dir = base_dir / split / "images"
        
        if not labels_dir.exists():
            continue

        for label_file in labels_dir.glob("*.txt"):
            img_file = None
            for ext in IMAGE_EXTENSIONS:
                potential_img = images_dir / f"{label_file.stem}{ext}"
                if potential_img.exists():
                    img_file = potential_img
                    break
            
            if img_file is None:
                continue

            new_lines = []
            with open(label_file, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    
                    orig_cls_id = int(parts[0])
                    if orig_cls_id in idx_to_new_id:
                        new_cls_id = idx_to_new_id[orig_cls_id]
                        parts[0] = str(new_cls_id)
                        new_lines.append(" ".join(parts))
                        stats[split][CLASS_NAMES[new_cls_id]] += 1

            if new_lines:
                unique_name = f"{ds_prefix.replace(' ', '_')}_{label_file.stem}"
                
                out_img_path = OUTPUT_DIR / "images" / split / f"{unique_name}{img_file.suffix}"
                out_lbl_path = OUTPUT_DIR / "labels" / split / f"{unique_name}.txt"
                
                shutil.copy2(img_file, out_img_path)
                
                with open(out_lbl_path, "w") as f:
                    f.write("\n".join(new_lines) + "\n")
                
                processed_files += 1

Starting dataset processing and class consolidation...


In [15]:
print(f"\nProcessing Complete! Total image/label pairs consolidated: {processed_files}")


Processing Complete! Total image/label pairs consolidated: 0


In [16]:
yaml_content = {
    "path": str(OUTPUT_DIR),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "names": [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]   # ← THE CHANGE
}

yaml_path = OUTPUT_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"Generated YOLOv11 data.yaml at: {yaml_path}")

Generated YOLOv11 data.yaml at: /kaggle/working/unified_dataset/data.yaml


In [17]:
print("\n" + "="*55)
print(" CONSOLIDATED CLASS INSTANCE SUMMARY")
print("="*55)

df_stats = pd.DataFrame(stats).fillna(0).astype(int)
df_stats["Total"] = df_stats.sum(axis=1)
print(df_stats.to_string())


 CONSOLIDATED CLASS INSTANCE SUMMARY
Empty DataFrame
Columns: [train, valid, test, Total]
Index: []


In [18]:
random.seed(42)

In [19]:
TEMP_DIR = Path("/kaggle/working/temp_dataset")
OUTPUT_DIR = Path("/kaggle/working/reorganized_dataset")

In [20]:
SPLIT_RATIOS = {"train": 0.8, "valid": 0.1, "test": 0.1}

In [21]:
def find_dataset_dir(possible_names):
    for name in possible_names:
        p = Path(f"/kaggle/working/{name}")
        if p.exists():
            return p
    for name in possible_names:
        clean = name.split("-")[0]
        matches = list(Path("/kaggle/working").glob(f"*{clean}*"))
        if matches:
            return matches[0]
    return Path(f"/kaggle/working/{possible_names[0]}")

DATASETS = {
    "Action(Stanford40)": {
        "base_dir": find_dataset_dir(["Action(Stanford40)-1", "Action(Stanford40)-2"]),
        "names": [
            "applauding", "drinking", "jumping", "looking_through_a_telescope",
            "phoning", "reading", "running", "smoking", "taking_photos",
            "texting_message", "waving_hands", "writing_on_a_book"
        ],
    },
    "Human-Action-Recognition": {
        "base_dir": find_dataset_dir(["Human-Action-Recognition-1", "Human-Action-Recognition-2"]),
        "names": [
            "Drinking", "Fall-Detected", "Fall_down", "Lying_down", "Nearly_fall",
            "Sit Down", "Sitting", "Standing", "Walking", "Walking_on_Stairs",
            "crawling", "falling", "sitting", "standing", "walking"
        ],
    },
    "sitting-standing-lyingv2": {
        "base_dir": find_dataset_dir(["sitting-standing-lyingv2-1", "sitting-standing-lyingv2-6"]),
        "names": ["lying", "sitting", "standing"],
    },
}

# Auto-update class names from data.yaml if available
for ds_key, cfg in DATASETS.items():
    yaml_p = cfg["base_dir"] / "data.yaml"
    if yaml_p.exists():
        try:
            with open(yaml_p, "r") as yf:
                yd = yaml.safe_load(yf)
                if "names" in yd:
                    if isinstance(yd["names"], list):
                        cfg["names"] = yd["names"]
                    elif isinstance(yd["names"], dict):
                        cfg["names"] = [yd["names"][k] for k in sorted(yd["names"].keys())]
        except Exception:
            pass


In [22]:
MAPPING_RULES = {
    "Fall-Detected": 0, "falling": 0, "Fall_down": 0, "Nearly_fall": 0,
    "sitting": 1, "Sitting": 1, "Sit Down": 1,
    "standing": 2, "Standing": 2,
    "walking": 3, "Walking": 3, "Walking_on_Stairs": 3, "running": 3, "jumping": 3,
    "lying": 4, "Lying_down": 4, "crawling": 4,
    "phoning": 5, "texting_message": 5, "taking_photos": 5,
    "reading": 6, "writing_on_a_book": 6, "drinking": 6, "Drinking": 6, "smoking": 6,
    "applauding": 7, "waving_hands": 7, "looking_through_a_telescope": 7
}

In [23]:
CLASS_NAMES = {
    0: "fall", 1: "sitting", 2: "standing", 3: "walking_running",
    4: "lying", 5: "phone_interaction", 6: "desk_activity", 7: "gestures"
}

In [24]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

In [25]:
if TEMP_DIR.exists():
    shutil.rmtree(TEMP_DIR)
(TEMP_DIR / "images").mkdir(parents=True, exist_ok=True)
(TEMP_DIR / "labels").mkdir(parents=True, exist_ok=True)


In [26]:
print("Consolidating all source datasets into temporary pool with Remapping...")
source_splits = ["train", "valid", "test"]
consolidated_count = 0

for ds_prefix, config in DATASETS.items():
    base_dir = config["base_dir"]
    orig_class_names = config["names"]
    
    idx_to_new_id = {
        orig_idx: MAPPING_RULES[orig_name]
        for orig_idx, orig_name in enumerate(orig_class_names)
        if orig_name in MAPPING_RULES
    }
    
    print(f"Processing {ds_prefix} from: {base_dir.name} (mapped {len(idx_to_new_id)}/{len(orig_class_names)} classes)")

    for split in source_splits:
        labels_dir = base_dir / split / "labels"
        images_dir = base_dir / split / "images"
        
        if not labels_dir.exists():
            continue

        for label_file in labels_dir.glob("*.txt"):
            img_file = next((images_dir / f"{label_file.stem}{ext}" for ext in IMAGE_EXTENSIONS if (images_dir / f"{label_file.stem}{ext}").exists()), None)
            if img_file is None:
                continue

            new_lines = []
            with open(label_file, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        orig_cls_id = int(parts[0])
                        if orig_cls_id in idx_to_new_id:
                            new_cls_id = idx_to_new_id[orig_cls_id]
                            parts[0] = str(new_cls_id)
                            new_lines.append(" ".join(parts))

            if new_lines:
                unique_name = f"{ds_prefix.replace(' ', '_')}_{split}_{label_file.stem}"
                shutil.copy2(img_file, TEMP_DIR / "images" / f"{unique_name}{img_file.suffix}")
                with open(TEMP_DIR / "labels" / f"{unique_name}.txt", "w") as f:
                    f.write("\n".join(new_lines) + "\n")
                consolidated_count += 1

print(f"\nConsolidation complete! Total image/label pairs remapped and pooled: {consolidated_count}")


Consolidating all source datasets into temporary pool with Remapping...
Processing Action(Stanford40) from: Action(Stanford40)-1 (mapped 12/12 classes)
Processing Human-Action-Recognition from: Human-Action-Recognition-1 (mapped 15/15 classes)
Processing sitting-standing-lyingv2 from: sitting-standing-lyingv2-1 (mapped 3/3 classes)

Consolidation complete! Total image/label pairs remapped and pooled: 14868


In [27]:
print("Performing dataset re-split (Train: 80%, Valid: 10%, Test: 10%)...")

#all_labels = list((TEMP_DIR / "labels").glob("*.txt"))

Performing dataset re-split (Train: 80%, Valid: 10%, Test: 10%)...


In [28]:
# 1. Collect all remapped labels from TEMP_DIR
all_labels = list((TEMP_DIR / "labels").glob("*.txt"))
print(f"Total remapped label files found in pool: {len(all_labels)}")

# 2. Extract primary class for balanced stratification
image_primary_classes = []
valid_label_files = []

for lbl_path in all_labels:
    with open(lbl_path, "r") as f:
        first_line = f.readline().strip()
        if first_line:
            primary_cls = int(first_line.split()[0])
            image_primary_classes.append(primary_cls)
            valid_label_files.append(lbl_path)

print(f"Total valid labels for splitting: {len(valid_label_files)}")


Total remapped label files found in pool: 14868
Total valid labels for splitting: 14868


In [29]:
train_files, temp_files, _, temp_classes = train_test_split(
    valid_label_files, image_primary_classes, test_size=0.2, random_state=42, stratify=image_primary_classes
)

In [30]:
val_files, test_files = train_test_split(
    temp_files, test_size=0.5, random_state=42, stratify=temp_classes
)

In [31]:
split_mapping = {
    "train": train_files,
    "valid": val_files,
    "test": test_files
}

stats = {s: Counter() for s in split_mapping.keys()}

In [32]:
from pathlib import Path
import shutil

# Ensure clean output directory
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split_name, files in split_mapping.items():
    out_img_dir = OUTPUT_DIR / "images" / split_name
    out_lbl_dir = OUTPUT_DIR / "labels" / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    for lbl_path in files:
        img_path = next((TEMP_DIR / "images" / f"{lbl_path.stem}{ext}" for ext in IMAGE_EXTENSIONS if (TEMP_DIR / "images" / f"{lbl_path.stem}{ext}").exists()), None)
        if img_path is None:
            continue

        # Copy image
        shutil.copy2(img_path, out_img_dir / img_path.name)
        
        # Read and record stats
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    if cls_id in CLASS_NAMES:
                        stats[split_name][CLASS_NAMES[cls_id]] += 1
                        
        # Copy remapped label
        shutil.copy2(lbl_path, out_lbl_dir / lbl_path.name)

print("Files successfully split and organized into train, valid, and test!")


Files successfully split and organized into train, valid, and test!


In [33]:
shutil.rmtree(TEMP_DIR)

In [34]:
yaml_content = {
    "path": str(OUTPUT_DIR),                                # or "./reorganized_dataset"
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "names": [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]  # ← THE CHANGE
}

yaml_path = OUTPUT_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"Dataset successfully created at: {OUTPUT_DIR}")

Dataset successfully created at: /kaggle/working/reorganized_dataset


In [35]:
print("\n" + "="*55)
print(" CONSOLIDATED CLASS INSTANCE SUMMARY")
print("="*55)

df_stats = pd.DataFrame(stats).fillna(0).astype(int)
df_stats["Total"] = df_stats.sum(axis=1)
print(df_stats.to_string())


 CONSOLIDATED CLASS INSTANCE SUMMARY
                   train  valid  test  Total
lying                920    112   114   1146
fall                4134    517   516   5167
walking_running     2262    278   269   2809
sitting             2910    359   372   3641
desk_activity        798    101   100    999
standing            2121    255   257   2633
gestures             559     73    69    701
phone_interaction    510     64    63    637


In [36]:
if Path("/kaggle/working/unified_dataset").exists():
    shutil.rmtree("/kaggle/working/unified_dataset")


In [37]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 76.5 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 5.6 MB/s eta 0:00:00


In [38]:
from ultralytics import YOLO

# 1. Load the YOLO11 nano model
model = YOLO('yolo11n.pt')

print("⏳ Starting YOLO11 Training...")

results = model.train(
    data = '/kaggle/working/reorganized_dataset/data.yaml',
    epochs=50,                           # Number of training loops
    imgsz=640,                           # Image resolution
    batch=16,                            # Batch size
    device=0,                            # 0 means use the GPU
    project='/kaggle/working/YOLO11_Results', 
    name='train_multi_class'
)

print("🎉 Training Completed Successfully!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
⏳ Starting YOLO11 Training...
Ultralytics 8.4.153 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/reorganized_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5,

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


       2/50      3.64G      1.239       1.94      1.448         19        640: 100% ━━━━━━━━━━━━ 744/744 6.9it/s 1:470.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 5.7it/s 8.2s0.1s
                   all       1487       1759      0.329       0.52      0.422      0.272

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      3.64G      1.276      1.806      1.476         17        640: 100% ━━━━━━━━━━━━ 744/744 6.9it/s 1:480.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 5.7it/s 8.2s0.1s
                   all       1487       1759       0.51      0.455      0.472      0.296

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      3.64G      1.262      1.723      1.462         13        640: 100% ━━━━━━━━━━━━ 744/744 7.0it/s 1:460.1sss
                 Class     Ima